In [ ]:
from pathlib import Path
import sys
import gc
import time
import random

import numpy as np
import pandas as pd
import torch

cwd = Path.cwd()

if (cwd / "common.py").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "common.py").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "common.py not found in the current folder or its parent. "
        "Set PROJECT_ROOT manually."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ------------------------------------------------------------
# Canonical project imports
# ------------------------------------------------------------
from common import LANGS, load_benchmark, record, RUN_DIR, RES_DIR

# ------------------------------------------------------------
# Settings (unchanged from the original BGE-M3 notebook)
# ------------------------------------------------------------
MODEL_NAME = "BAAI/bge-m3"
MODEL_TAG = "bge-m3"          # folder name under runs/
SEED = 42
BATCH_SIZE = 8
MAX_SEQ_LENGTH = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Output directory for this model
BGE_RUN_DIR = RUN_DIR / MODEL_TAG
BGE_RUN_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------
print("Project root :", PROJECT_ROOT)
print("Runs dir     :", BGE_RUN_DIR)
print("Results dir  :", RES_DIR)
print("PyTorch      :", torch.__version__)
print("Device       :", DEVICE)
print("Model        :", MODEL_NAME)
print("Batch size   :", BATCH_SIZE)
print("Max seq len  :", MAX_SEQ_LENGTH)
print("Seed         :", SEED)
print("Languages    :", len(LANGS), LANGS)

assert len(LANGS) == 13

# Which languages already have BGE-M3 scores?
done = [l for l in LANGS if (BGE_RUN_DIR / f"{l}_scores.npy").exists()]
print("Already done :", done if done else "none")

Project root : c:\Users\sanka\OneDrive\Desktop\research paper
Runs dir     : C:\Users\sanka\OneDrive\Desktop\research paper\runs\bge-m3
Results dir  : C:\Users\sanka\OneDrive\Desktop\research paper\results
PyTorch      : 2.14.1+cpu
Device       : cpu
Model        : BAAI/bge-m3
Batch size   : 8
Max seq len  : 512
Seed         : 42
Languages    : 13 ['as', 'bn', 'gu', 'hi', 'kn', 'ml', 'mr', 'ne', 'or', 'pa', 'ta', 'te', 'ur']
Already done : ['as']


In [3]:
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-m3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8
MAX_SEQ_LENGTH = 512

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

model.max_seq_length = MAX_SEQ_LENGTH

print("Model loaded:", MODEL_NAME)
print("Device:", DEVICE)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded: BAAI/bge-m3
Device: cpu
Embedding dimension: 1024


C:\Users\sanka\AppData\Local\Temp\ipykernel_38688\477637088.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [5]:
lang = "hi"

queries, corpus, gold = load_benchmark(lang)

print("Queries :", len(queries))
print("Corpus  :", len(corpus))
print("Gold    :", gold.shape, gold.dtype)
print("Query columns :", list(queries.columns))
print("Corpus columns:", list(corpus.columns))

# Canonical invariants
assert {"query_id", "query", "gold_doc_id"}.issubset(queries.columns)
assert {"doc_id", "text"}.issubset(corpus.columns)
assert len(gold) == len(queries)
assert gold.min() >= 0 and gold.max() < len(corpus)
assert (corpus["doc_id"].iloc[gold].to_numpy() == queries["gold_doc_id"].to_numpy()).all()
print("OK: gold indices map to the correct gold_doc_id")

# Small subset: first 16 queries and their gold documents
N = 16
sub_queries = queries["query"].tolist()[:N]
sub_gold_idx = gold[:N]
sub_docs = corpus["text"].iloc[sub_gold_idx].tolist()

t0 = time.time()

q_emb = model.encode(
    sub_queries,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=False,
    convert_to_numpy=True
)

d_emb = model.encode(
    sub_docs,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=False,
    convert_to_numpy=True
)

print(f"Encoding time for {N} queries + {N} docs: {time.time() - t0:.1f} sec")
print("Query emb:", q_emb.shape, "| Doc emb:", d_emb.shape)

assert np.allclose(np.linalg.norm(q_emb, axis=1), 1.0, atol=1e-5)
assert np.allclose(np.linalg.norm(d_emb, axis=1), 1.0, atol=1e-5)

# Mini retrieval: each query against only these N documents
S_small = q_emb @ d_emb.T
top1 = S_small.argmax(axis=1)
print("Mini top-1 accuracy (pool of 16):", float((top1 == np.arange(N)).mean()))
print("Sanity check passed.")

del q_emb, d_emb, S_small
gc.collect()

Queries : 1000
Corpus  : 1000
Gold    : (1000,) int64
Query columns : ['query_id', 'query', 'gold_doc_id']
Corpus columns: ['doc_id', 'text']
OK: gold indices map to the correct gold_doc_id
Encoding time for 16 queries + 16 docs: 8.8 sec
Query emb: (16, 1024) | Doc emb: (16, 1024)
Mini top-1 accuracy (pool of 16): 0.875
Sanity check passed.


0

In [ ]:
def evaluate_language(lang):
    scores_path = BGE_RUN_DIR / f"{lang}_scores.npy"

    if scores_path.exists():
        print(f"[{lang}] SKIP: {scores_path.name} already exists")
        return None

    print("\n" + "=" * 70)
    print(f"Evaluating: {lang}")
    print("=" * 70)

    total_start = time.time()

    queries, corpus, gold = load_benchmark(lang)

    documents = corpus["text"].tolist()
    query_texts = queries["query"].tolist()

    print(f"Queries: {len(query_texts)}")
    print(f"Corpus : {len(documents)}")

    assert len(gold) == len(query_texts)
    assert gold.min() >= 0 and gold.max() < len(documents)

    start = time.time()

    corpus_embeddings = model.encode(
        documents,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    corpus_time = time.time() - start

    print(f"Corpus embeddings: {corpus_embeddings.shape}")
    print(f"Corpus encoding time: {corpus_time / 60:.2f} min")

    assert corpus_embeddings.shape == (len(documents), 1024)
    assert np.allclose(np.linalg.norm(corpus_embeddings, axis=1), 1.0, atol=1e-5)

    start = time.time()

    query_embeddings = model.encode(
        query_texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    query_time = time.time() - start

    print(f"Query embeddings: {query_embeddings.shape}")
    print(f"Query encoding time: {query_time / 60:.2f} min")

    assert query_embeddings.shape == (len(query_texts), 1024)
    assert np.allclose(np.linalg.norm(query_embeddings, axis=1), 1.0, atol=1e-5)

    similarities = query_embeddings @ corpus_embeddings.T

    print(f"Similarity matrix: {similarities.shape}")

    assert similarities.shape == (len(query_texts), len(documents))

    m = record(
        model=MODEL_TAG,
        key=lang,
        S=similarities,
        gold=gold,
        setting="mono"
    )

    total_time = time.time() - total_start

    print("\n" + "-" * 70)
    print(f"RESULT: {lang}")
    print("-" * 70)
    print(f"MRR:       {m['MRR']:.4f}")
    print(f"MRR@10:    {m['MRR@10']:.4f}")
    print(f"Recall@1:  {m['R@1']:.4f}")
    print(f"Recall@5:  {m['R@5']:.4f}")
    print(f"Recall@10: {m['R@10']:.4f}")
    print(f"Recall@20: {m['R@20']:.4f}")
    print(f"Recall@100:{m['R@100']:.4f}")
    print(f"Total time: {total_time / 60:.2f} min")
    print("-" * 70)

    # Free memory
    del corpus_embeddings, query_embeddings, similarities
    gc.collect()

    return m


print("evaluate_language() defined.")

evaluate_language() defined.


In [7]:
all_metrics = []
failed = []

for lang in LANGS:
    try:
        m = evaluate_language(lang)
        if m is not None:
            all_metrics.append(m)
    except Exception as e:
        print("\n" + "!" * 70)
        print(f"FAILED: {lang}")
        print(f"ERROR : {repr(e)}")
        print("!" * 70)
        failed.append(lang)
        continue

print("\n" + "=" * 70)
print("RUN FINISHED")
print("=" * 70)
print("Newly computed this session:", [m["key"] for m in all_metrics])
print("Failed                     :", failed if failed else "none")

[as] SKIP: as_scores.npy already exists

Evaluating: bn
Queries: 1000
Corpus : 1000


Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Corpus embeddings: (1000, 1024)
Corpus encoding time: 14.57 min


Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Query embeddings: (1000, 1024)
Query encoding time: 2.14 min
Similarity matrix: (1000, 1000)

----------------------------------------------------------------------
RESULT: bn
----------------------------------------------------------------------
MRR:       0.8602
MRR@10:    0.8581
Recall@1:  0.8110
Recall@5:  0.9130
Recall@10: 0.9430
Recall@20: 0.9640
Recall@100:0.9860
Total time: 16.71 min
----------------------------------------------------------------------

Evaluating: gu
Queries: 999
Corpus : 999


...

Query embeddings: (1000, 1024)
Query encoding time: 3.54 min
Similarity matrix: (1000, 1000)

----------------------------------------------------------------------
RESULT: te
----------------------------------------------------------------------
MRR:       0.8611
MRR@10:    0.8593
Recall@1:  0.8120
Recall@5:  0.9190
Recall@10: 0.9420
Recall@20: 0.9580
Recall@100:0.9830
Total time: 29.37 min
----------------------------------------------------------------------

Evaluating: ur
Queries: 1000
Corpus : 1000


Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Corpus embeddings: (1000, 1024)
Corpus encoding time: 18.13 min


Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Query embeddings: (1000, 1024)
Query encoding time: 2.40 min
Similarity matrix: (1000, 1000)

----------------------------------------------------------------------
RESULT: ur
----------------------------------------------------------------------
MRR:       0.8365
MRR@10:    0.8337
Recall@1:  0.7810
Recall@5:  0.9010
Recall@10: 0.9200
Recall@20: 0.9490
Recall@100:0.9790
Total time: 20.54 min
----------------------------------------------------------------------

RUN FINISHED
Newly computed this session: ['bn', 'gu', 'hi', 'kn', 'ml', 'mr', 'ne', 'or', 'pa', 'ta', 'te', 'ur']
Failed                     : none


In [8]:
from common import RES_DIR

problems = []

for lang in LANGS:
    sp = BGE_RUN_DIR / f"{lang}_scores.npy"
    rp = BGE_RUN_DIR / f"{lang}_ranks.npy"

    if not sp.exists():
        problems.append(f"{lang}: missing scores file")
        continue
    if not rp.exists():
        problems.append(f"{lang}: missing ranks file (delete {sp.name} and re-run Cell 5)")
        continue

    S = np.load(sp, mmap_mode="r")
    r = np.load(rp)

    queries, corpus, gold = load_benchmark(lang)

    if S.shape != (len(queries), len(corpus)):
        problems.append(f"{lang}: scores shape {S.shape} != {(len(queries), len(corpus))}")
    if r.shape != (len(queries),):
        problems.append(f"{lang}: ranks shape {r.shape} != {(len(queries),)}")

for other in sorted(p for p in RUN_DIR.iterdir() if p.is_dir() and p.name != MODEL_TAG):
    for lang in LANGS:
        op = other / f"{lang}_scores.npy"
        sp = BGE_RUN_DIR / f"{lang}_scores.npy"
        if op.exists() and sp.exists():
            so = np.load(op, mmap_mode="r").shape
            sb = np.load(sp, mmap_mode="r").shape
            if so != sb:
                problems.append(f"{lang}: {other.name} shape {so} != bge-m3 shape {sb}")

print("Score files:", sum((BGE_RUN_DIR / f"{l}_scores.npy").exists() for l in LANGS), "/ 13")
print("Rank files :", sum((BGE_RUN_DIR / f"{l}_ranks.npy").exists() for l in LANGS), "/ 13")

if problems:
    print("\nPROBLEMS:")
    for p in problems:
        print(" -", p)
else:
    print("\nAll BGE-M3 score/rank files exist, shapes match the canonical benchmark")
    print("and match the other models' runs.")

mpath = RES_DIR / "metrics.csv"
met = pd.read_csv(mpath)
bge = met[(met["model"] == MODEL_TAG) & (met["setting"] == "mono")]

print("\nbge-m3 rows in metrics.csv:", len(bge), "(expected 13)")

summary = (
    bge.set_index("key")
    .reindex(LANGS)[["MRR", "MRR@10", "R@1", "R@5", "R@10", "R@20", "R@100", "n_queries", "n_docs"]]
)
print(summary.round(4).to_string())

Score files: 13 / 13
Rank files : 13 / 13

All BGE-M3 score/rank files exist, shapes match the canonical benchmark
and match the other models' runs.

bge-m3 rows in metrics.csv: 13 (expected 13)
        MRR  MRR@10     R@1     R@5    R@10   R@20  R@100  n_queries  n_docs
key                                                                         
as   0.8073  0.8042  0.7460  0.8750  0.9110  0.936  0.977       1000    1000
bn   0.8602  0.8581  0.8110  0.9130  0.9430  0.964  0.986       1000    1000
gu   0.8336  0.8316  0.7678  0.9139  0.9369  0.953  0.979        999     999
hi   0.8863  0.8849  0.8400  0.9440  0.9570  0.969  0.991       1000    1000
kn   0.8379  0.8356  0.7860  0.9000  0.9200  0.937  0.979       1000    1000
ml   0.8405  0.8381  0.7880  0.9000  0.9270  0.946  0.983       1000    1000
mr   0.8521  0.8503  0.8000  0.9170  0.9390  0.956  0.982       1000    1000
ne   0.8504  0.8487  0.7950  0.9210  0.9440  0.959  0.983       1000    1000
or   0.7953  0.7925  0.7310  0.8750